# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# The mlcroissant metadata object can be printed for basic info
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` field for consistency.

In [ ]:
# List all record sets by their @id
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}\n")

# For each record set, list its fields and columns by @id
for rs in record_sets:
    print(f"--- RecordSet '{rs.name}' (@id: {rs.id}) fields:")
    for field in rs.fields:
        print(f"    Field name: {field.name}, @id: {field.id}, dataType: {field.data_type}")
        if hasattr(field, 'column'):
            print(f"      Column(s): {getattr(field, 'column', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview above.

In [ ]:
# We'll extract data from the primary record set (assume only one main record set is present).
# Get the first record set's @id
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Loop over all record sets and load them into pandas DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} records from RecordSet '{record_set_id}'.")
    print(f"Columns: {dataframes[record_set_id].columns.tolist()[:10]} {'...' if len(dataframes[record_set_id].columns) > 10 else ''}\n")
# Pick the first record set for detailed exploration
main_rs_id = record_set_ids[0]
print(f"Preview of the first 5 rows from RecordSet @id={main_rs_id}:")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. Reference all fields by their `@id`.

In [ ]:
# Example: Filter by a numeric field and normalize it
# We'll try to pick a numeric field from the RecordSet based on available data types.
main_rs = [rs for rs in record_sets if rs.id == main_rs_id][0]
numeric_fields = [f for f in main_rs.fields if f.data_type in ('schema:Number', 'schema:Float', 'schema:Integer')]
if numeric_fields:
    numeric_field = numeric_fields[0].id  # Use @id as column name
    print(f"Using numeric field for analysis: {numeric_fields[0].name} (@id: {numeric_field})")
else:
    numeric_field = dataframes[main_rs_id].columns[0]  # fallback
    print(f"No explicit numeric field found, using column: {numeric_field}")

# Filter for records where the numeric field > threshold
threshold = 10
filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {filtered_df.shape[0]} rows\n")
# Normalize the numeric field
filtered_df.loc[:, f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field if available
categorical_fields = [f for f in main_rs.fields if f.data_type == 'schema:Text']
if categorical_fields:
    group_field = categorical_fields[0].id  # Use @id as column name
    print(f"\nGrouping by: {categorical_fields[0].name} (@id: {group_field})\n")
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field, f"{numeric_field}_normalized"].mean()
        print("Grouped means (top 5 groups):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Visualize the distribution of the main numeric field in the filtered DataFrame
plt.figure(figsize=(8,4))
filtered_df[numeric_field].hist(bins=30)
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.title(f'Distribution of {numeric_field} (> {threshold})')
plt.show()

# If a group_field is available, show a boxplot
if categorical_fields and group_field in filtered_df.columns and len(filtered_df[group_field].unique()) < 20:
    plt.figure(figsize=(12,6))
    filtered_df.boxplot(column=numeric_field, by=group_field, grid=False, rot=90)
    plt.title(f'{numeric_field} by {group_field}')
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load a FAIR^2 ML Croissant dataset by schema URL
- Inspect its logical structure referencing all objects by their `@id`
- Extract records from the main data table(s)
- Filter, normalize, and group numeric data fields
- Visualize distributions and grouped data

**Tips for further analysis:**
- Explore the full set of available fields and record sets using their `@id` as shown
- Extend the EDA to other numeric or categorical fields as relevant for your analysis
- Use the Croissant logical schema for reproducible workflows referencing `@id` consistently